In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

In [4]:
matches['date']=pd.to_datetime(matches['date'])

matches['date'].head()

### checking duplicates ###

In [5]:
matches.duplicated().sum()

np.int64(0)

In [6]:
deliveries.duplicated().sum()

np.int64(0)

No duplicates found 
if found : 
matches.drop_duplicates(inplace=True)
deliveries.drop_duplicates(inplace=True)

### Handling Null values ###

In [7]:
matches.isnull().sum()


id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

In [8]:
# Fill missing city values
matches['city'] = (
    matches['city']
    .fillna('Unknown')
)

# Fill missing player of match
matches['player_of_match'] = (
    matches['player_of_match']
    .fillna('No Award')
)

# Fill result margin
matches['result_margin'] = (
    matches['result_margin']
    .fillna(0)
)

# Fill matches with no winner
matches['winner'] = (
    matches['winner']
    .fillna('No Result')
)

# Fill target overs
matches['target_overs'] = (
    matches['target_overs']
    .fillna(20)
)

# Fill method
matches['method'] = (
    matches['method']
    .fillna('Normal')
)



In [9]:
matches.isnull().sum()

id                 0
season             0
city               0
date               0
match_type         0
player_of_match    0
venue              0
team1              0
team2              0
toss_winner        0
toss_decision      0
winner             0
result             0
result_margin      0
target_runs        3
target_overs       0
super_over         0
method             0
umpire1            0
umpire2            0
dtype: int64

All the NULL values in match data set are handled 

In [10]:
deliveries.isnull().sum()

match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

In [11]:
deliveries['extras_type'] = (
    deliveries['extras_type']
    .fillna('No Extra')
)

deliveries['player_dismissed'] = (
    deliveries['player_dismissed']
    .fillna('Not Out')
)

deliveries['dismissal_kind'] = (
    deliveries['dismissal_kind']
    .fillna('No Wicket')
)

deliveries['fielder'] = (
    deliveries['fielder']
    .fillna('No Fielder')
)

In [12]:
deliveries.isnull().sum()

match_id            0
inning              0
batting_team        0
bowling_team        0
over                0
ball                0
batter              0
bowler              0
non_striker         0
batsman_runs        0
extra_runs          0
total_runs          0
extras_type         0
is_wicket           0
player_dismissed    0
dismissal_kind      0
fielder             0
dtype: int64

### delhi  name chnaged ###

In [51]:
matches['team1'] = matches['team1'].replace(
    'Delhi Daredevils',
    'Delhi Capitals'
)

matches['team2'] = matches['team2'].replace(
    'Delhi Daredevils',
    'Delhi Capitals'
)

matches['winner'] = matches['winner'].replace(
    'Delhi Daredevils',
    'Delhi Capitals'
)

matches['toss_winner'] = matches['toss_winner'].replace(
    'Delhi Daredevils',
    'Delhi Capitals'
)

### punjab name change ###

In [14]:
matches['team1'] = matches['team1'].replace(
    'Kings XI Punjab',
    'Punjab Kings'
)

matches['team2'] = matches['team2'].replace(
    'Kings XI Punjab',
    'Punjab Kings'
)

matches['winner'] = matches['winner'].replace(
    'Kings XI Punjab',
    'Punjab Kings'
)

matches['toss_winner'] = matches['toss_winner'].replace(
    'Kings XI Punjab',
    'Punjab Kings'
)

### player name cleaning ###

In [15]:
matches['player_of_match'] = (
    matches['player_of_match']
    .str.strip()
    .str.title()
)

### total runs per ball ###

In [16]:
deliveries['total_runs_ball'] = (
    deliveries['batsman_runs']
    + deliveries['extra_runs']
)

### powerplay flag ###

In [17]:
deliveries['is_powerplay'] = (
    deliveries['over']
    .apply(lambda x: 1 if x <= 6 else 0)
)

### Death over flag ###

In [18]:
deliveries['is_death_over'] = (
    deliveries['over']
    .apply(lambda x: 1 if x >= 16 else 0)
)

### Season play ###

In [19]:
matches['season_year'] = (
    matches['date'].dt.year
)

In [20]:
first_innings = deliveries[
    deliveries['inning'] == 1
]

In [21]:
first_innings_score = (
    first_innings
    .groupby('match_id')['total_runs_ball']
    .sum()
    .reset_index()
)

In [22]:
first_innings_score.rename(
    columns={
        'total_runs_ball': 'first_innings_score'
    },
    inplace=True
)

In [23]:
first_innings_score['target_runs'] = (
    first_innings_score['first_innings_score'] + 1
)

In [24]:
matches = matches.merge(
    first_innings_score[
        ['match_id', 'target_runs']
    ],
    left_on='id',
    right_on='match_id',
    how='left'
)

In [25]:
matches['target_overs'] = 20

In [29]:
matches['required_run_rate'] = (
    matches.get('target_runs', 0)  # Returns 0 if column doesn't exist
    / matches.get('target_overs', 1)  # Returns 1 to avoid division by zero
)

In [30]:
matches.isnull().sum()

id                   0
season               0
city                 0
date                 0
match_type           0
player_of_match      0
venue                0
team1                0
team2                0
toss_winner          0
toss_decision        0
winner               0
result               0
result_margin        0
target_runs_x        3
target_overs         0
super_over           0
method               0
umpire1              0
umpire2              0
season_year          0
match_id             0
target_runs_y        0
required_run_rate    0
dtype: int64

In [32]:
deliveries.isnull().sum()

match_id            0
inning              0
batting_team        0
bowling_team        0
over                0
ball                0
batter              0
bowler              0
non_striker         0
batsman_runs        0
extra_runs          0
total_runs          0
extras_type         0
is_wicket           0
player_dismissed    0
dismissal_kind      0
fielder             0
total_runs_ball     0
is_powerplay        0
is_death_over       0
dtype: int64

In [33]:
matches[['target_runs_x', 'target_runs_y']].head(10)

,target_runs_x,target_runs_y
0,223.0,223
1,241.0,241
2,130.0,130
3,166.0,166
4,111.0,111
5,167.0,167
6,143.0,143
7,209.0,209
8,215.0,215
9,183.0,183


In [34]:
matches.drop(columns=['target_runs_x'], inplace=True)

matches.rename(
    columns={'target_runs_y': 'target_runs'},
    inplace=True
)

In [35]:
matches.isnull().sum()

id                   0
season               0
city                 0
date                 0
match_type           0
player_of_match      0
venue                0
team1                0
team2                0
toss_winner          0
toss_decision        0
winner               0
result               0
result_margin        0
target_overs         0
super_over           0
method               0
umpire1              0
umpire2              0
season_year          0
match_id             0
target_runs          0
required_run_rate    0
dtype: int64

In [37]:
matches.shape

(1095, 23)

In [38]:
deliveries.shape

(260920, 20)

In [39]:
matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 1095 non-null   int64         
 1   season             1095 non-null   object        
 2   city               1095 non-null   object        
 3   date               1095 non-null   datetime64[ns]
 4   match_type         1095 non-null   object        
 5   player_of_match    1095 non-null   object        
 6   venue              1095 non-null   object        
 7   team1              1095 non-null   object        
 8   team2              1095 non-null   object        
 9   toss_winner        1095 non-null   object        
 10  toss_decision      1095 non-null   object        
 11  winner             1095 non-null   object        
 12  result             1095 non-null   object        
 13  result_margin      1095 non-null   float64       
 14  target_o

In [40]:
deliveries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   match_id          260920 non-null  int64 
 1   inning            260920 non-null  int64 
 2   batting_team      260920 non-null  object
 3   bowling_team      260920 non-null  object
 4   over              260920 non-null  int64 
 5   ball              260920 non-null  int64 
 6   batter            260920 non-null  object
 7   bowler            260920 non-null  object
 8   non_striker       260920 non-null  object
 9   batsman_runs      260920 non-null  int64 
 10  extra_runs        260920 non-null  int64 
 11  total_runs        260920 non-null  int64 
 12  extras_type       260920 non-null  object
 13  is_wicket         260920 non-null  int64 
 14  player_dismissed  260920 non-null  object
 15  dismissal_kind    260920 non-null  object
 16  fielder           260920 non-null  obj

In [42]:
matches['required_run_rate'] = (
    matches['target_runs'] / matches['target_overs']
)

In [43]:
matches[['target_runs',
         'target_overs',
         'required_run_rate',
         'season_year']].head()

,target_runs,target_overs,required_run_rate,season_year
0,223,20,11.15,2008
1,241,20,12.05,2008
2,130,20,6.50,2008
3,166,20,8.30,2008
4,111,20,5.55,2008


In [44]:
deliveries[['total_runs_ball',
            'is_powerplay',
            'is_death_over']].head()

,total_runs_ball,is_powerplay,is_death_over
0,1,1,0
1,0,1,0
2,1,1,0
3,0,1,0
4,0,1,0


In [48]:
import os

print(os.getcwd())

C:\Users\91939\anaconda_projects\IPL_team_analysis


In [47]:
import os

print(os.listdir())

['.ipynb_checkpoints', '01_data_loading.ipynb', '02_data_cleaning.ipynb', 'data', 'deliveries.csv', 'matches.csv']


In [ ]:
matches.to_csv(
    'data/cleaned_matches.csv',
    index=False
)

deliveries.to_csv(
    'data/cleaned_deliveries.csv',
    index=False
)

In [50]:
import os
print(os.listdir('data'))

['cleaned_deliveries.csv', 'cleaned_matches.csv']
